# Yotuubef Studio

Documentary pipeline control panel built on **Gradio**. The entire pipeline (idea generation, scripting, render) is driven from a single web UI accessed via a public Colab link.

Before running:
1. Set **Runtime, Change runtime type, GPU**.
2. Add Colab Secrets (left sidebar key icon): `REDDIT_CLIENT_ID`, `REDDIT_CLIENT_SECRET`, `NVIDIA_NIM_API_KEY`, `HACKCLUB_SEARCH_API_KEY`, `YOUTUBE_TOKEN_JSON` (optional).
3. Put one or more `.mp4` background videos in your Drive folder (path shown in Cell 2 output).

Workflow:
- **Cell 1**: Environment setup (one-time per session).
- **Cell 2**: Mount Drive and link persistent folders.
- **Cell 3**: Secrets and preflight checks.
- **Cell 4**: Launch the Gradio GUI. Click the printed public URL to open the studio.

Inside the GUI:
- **Dashboard** - live pipeline status, project list, log stream (auto-refreshes every 3s).
- **New Run** - start a new documentary, stream live logs.
- **Script Editor** - review/edit segments before rendering.
- **History** - browse past run results and upload database.
- **Settings** - edit `config.yaml` (model, TTS, codecs, subreddits).
- **Results** - preview rendered videos and list artefacts.

In [ ]:
# Cell 1: Environment Setup & Dependencies
import os, shlex, subprocess
from pathlib import Path

def run(cmd, allow_fail=False):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, text=True)
    if r.returncode != 0 and not allow_fail:
        raise RuntimeError(f"Command failed ({r.returncode}): {cmd}")

# 1. System dependencies
run("apt-get -qq update")
run("apt-get -qq install -y ffmpeg sox git gcc g++")

# 2. Clone repo
REPO_URL = "https://github.com/beenycool/yotuubef.git"
REPO_DIR = Path("/content/yotuubef")
if not REPO_DIR.exists():
    run(f"git clone {shlex.quote(REPO_URL)} {shlex.quote(str(REPO_DIR))}")
os.chdir(REPO_DIR)
run("git pull --ff-only || true", allow_fail=True)

# 3. Python dependencies (includes gradio from requirements.txt)
run("python3 -m pip install -q --upgrade pip setuptools wheel")
run("python3 -m pip install -q -r requirements.txt")
run("python3 -m pip install -q nest_asyncio")

# 4. Qwen3-TTS (optional; TTS falls back to ElevenLabs if missing)
qwen_cmd = "python3 -m pip install -q git+https://github.com/QwenLM/Qwen3-TTS.git"
print(f"$ {qwen_cmd}")
if subprocess.run(qwen_cmd, shell=True, text=True).returncode != 0:
    print("Warning: Qwen3-TTS install failed. Pipeline falls back to ElevenLabs.")

print("Setup complete.")

In [ ]:
# Cell 2: Mount Drive & Configure Persistence
import os, shutil
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

REPO_DIR = Path("/content/yotuubef")
os.chdir(REPO_DIR)

DRIVE_ROOT = Path("/content/drive/MyDrive/yotuubef")
PERSIST_ROOT = DRIVE_ROOT / "persistent"
RUNS_ROOT = DRIVE_ROOT / "runs"
for p in [DRIVE_ROOT, PERSIST_ROOT, RUNS_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

BACKGROUND_FOLDER_PATH = PERSIST_ROOT / "backgrounds"
BACKGROUND_FOLDER_PATH.mkdir(parents=True, exist_ok=True)
TOKEN_FILE_PATH = PERSIST_ROOT / "tokens" / "youtube_token.json"
TOKEN_FILE_PATH.parent.mkdir(parents=True, exist_ok=True)

important_dirs = {
    "findings":    PERSIST_ROOT / "findings",
    "processed":   PERSIST_ROOT / "processed",
    "results":      PERSIST_ROOT / "results",
    "logs":         PERSIST_ROOT / "logs",
    "db":           PERSIST_ROOT / "databases",
}
for p in important_dirs.values():
    p.mkdir(parents=True, exist_ok=True)

def relink_dir(local_path, drive_path):
    local_path.parent.mkdir(parents=True, exist_ok=True)
    if local_path.is_symlink():
        if local_path.resolve(strict=False) == drive_path.resolve(strict=False):
            return
        local_path.unlink()
    elif local_path.exists():
        if not local_path.is_dir():
            raise RuntimeError(f"Cannot relink non-directory: {local_path}")
        for item in local_path.iterdir():
            tgt = drive_path / item.name
            if tgt.exists():
                raise RuntimeError(f"Target already exists: {tgt}")
        shutil.rmtree(local_path)
    local_path.symlink_to(drive_path, target_is_directory=True)

link_map = {
    REPO_DIR / "findings":           important_dirs["findings"],
    REPO_DIR / "processed":          important_dirs["processed"],
    REPO_DIR / "logs":               important_dirs["logs"],
    REPO_DIR / "data" / "results":   important_dirs["results"],
    REPO_DIR / "data" / "logs":      important_dirs["logs"],
    REPO_DIR / "data" / "databases": important_dirs["db"],
}
for local_dir, drive_dir in link_map.items():
    relink_dir(local_dir, drive_dir)

print("Drive mounted and persistence linked.")
print(f"Background folder: {BACKGROUND_FOLDER_PATH}")

In [ ]:
# Cell 3: Secrets & Preflight Check
import json, os
from pathlib import Path

REPO_DIR = Path("/content/yotuubef")
BACKGROUND_FOLDER_PATH = Path("/content/drive/MyDrive/yotuubef/persistent/backgrounds")
TOKEN_FILE_PATH = Path("/content/drive/MyDrive/yotuubef/persistent/tokens/youtube_token.json")

def get_secret(name, default=""):
    try:
        from google.colab import userdata
        v = userdata.get(name)
        return v if v is not None else default
    except Exception:
        return os.getenv(name, default)

secret_map = {
    "REDDIT_CLIENT_ID":          "REDDIT_CLIENT_ID",
    "REDDIT_CLIENT_SECRET":       "REDDIT_CLIENT_SECRET",
    "NVIDIA_NIM_API_KEY":         "NVIDIA_NIM_API_KEY",
    "HACKCLUB_SEARCH_API_KEY":    "HACKCLUB_SEARCH_API_KEY",
    "YOUTUBE_TOKEN_JSON":         "YOUTUBE_TOKEN_JSON",
}
for secret_name, env_name in secret_map.items():
    val = get_secret(secret_name, "")
    if val:
        os.environ[env_name] = val

os.environ["BACKGROUND_FOLDER"] = str(BACKGROUND_FOLDER_PATH)
os.environ["YOUTUBE_TOKEN_FILE"] = str(TOKEN_FILE_PATH)

youtube_token_json = get_secret("YOUTUBE_TOKEN_JSON", "")
if youtube_token_json:
    try:
        parsed = json.loads(youtube_token_json)
        TOKEN_FILE_PATH.parent.mkdir(parents=True, exist_ok=True)
        TOKEN_FILE_PATH.write_text(json.dumps(parsed), encoding="utf-8")
        print(f"Wrote YouTube token file to: {TOKEN_FILE_PATH}")
    except Exception as exc:
        print(f"Warning: Could not parse YOUTUBE_TOKEN_JSON ({exc})")

# PREFLIGHT
print("Running Preflight Checks...")
errors = []
if not os.environ.get("REDDIT_CLIENT_ID"):      errors.append("REDDIT_CLIENT_ID missing")
if not os.environ.get("REDDIT_CLIENT_SECRET"):   errors.append("REDDIT_CLIENT_SECRET missing")
if not os.environ.get("NVIDIA_NIM_API_KEY"):    errors.append("NVIDIA_NIM_API_KEY missing")
if not os.environ.get("HACKCLUB_SEARCH_API_KEY"): errors.append("HACKCLUB_SEARCH_API_KEY missing")

bg_videos = list(BACKGROUND_FOLDER_PATH.glob("*.mp4")) if BACKGROUND_FOLDER_PATH.exists() else []
if not bg_videos:
    errors.append(f"No .mp4 background videos in {BACKGROUND_FOLDER_PATH}")

if errors:
    print("\n".join(errors))
    raise RuntimeError("Preflight checks failed.")
print("All credentials present.")
print(f"Found {len(bg_videos)} background videos.")
print("Preflight passed. Run the next cell to launch the GUI.")

## Cell 4: Launch the Gradio Studio

Import the GUI module and launch with `share=True`. Colab will tunnel a public URL - click it to open the studio in a new tab.

The cell blocks while the server is alive. To stop: **Runtime, Interrupt execution**.

In [ ]:
# Cell 4: Launch Gradio Studio
import os, sys, subprocess
from pathlib import Path

# Pin uvicorn for Python 3.14 compatibility (loop_factory kwarg fix)
subprocess.run("pip install -q uvicorn==0.30.6", shell=True)

# Kill any lingering uvicorn/gradio on target port from a previous run
# (Colab cells share the same kernel — ports persist after interrupt)
subprocess.run("fuser -k 7860/tcp 7861/tcp 2>/dev/null; sleep 0.5", shell=True)

REPO_DIR = Path("/content/yotuubef")
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# Colab runs an asyncio loop already; nest_asyncio lets us use asyncio.run() inside it.
import nest_asyncio
nest_asyncio.apply()

from gui.app import launch

print("Launching Yotuubef Studio...")
print("A public Gradio URL will appear below. Click it to open the GUI.")
launch(share=True, server_port=7860)